# CTAS reproducibility bundle: SN2025bbe multi-band Bazin fit

This notebook uses only the files in this folder. It re-selects the fit's input detections
from the published dossier by the run's stated input rule, checks them against the run's
recorded inputs, draws the light curve with the published model, refits independently and
compares. A phenomenological fit; it is not a classification, a physical model or a distance.

In [ ]:
import json, math
from pathlib import Path
import numpy as np
from scipy.optimize import least_squares
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

record = json.loads(Path("records/SN2025bbe.json").read_text())
config = json.loads(Path("analysis/config.json").read_text())
run = next(r for r in record["lightcurve_science"]["runs"] if r["analysisType"] == "multiband-bazin")
assert run["status"] == "complete" and run["methodVersion"] == config["methodVersion"]
inputs = record["lightcurve_science"]["inputs"]
print(record["name"], run["methodName"], run["methodVersion"], "reduced chi2", run["result"]["reducedChi2"])

In [ ]:
# The run's input rule: ZTF g/r/i difference-PSF detections, one row per exposure, AB,
# uncertainty <= 0.5 mag (this record has no ATLAS rows).
rows = {}
for o in record["follow_up"]["observations"]:
    if o.get("detection") == 1 and o.get("difference_photometry") == 1 and o.get("magnitude_system") == "AB" \
            and o.get("magnitude_error") is not None and o["magnitude_error"] <= 0.5 and o.get("band") in ("g", "r", "i"):
        rows.setdefault(("ztf" + o["band"], round(o["mjd"], 5)), o)
by_band = {}
for band, _ in rows:
    by_band[band] = by_band.get(band, 0) + 1
print("selected", len(rows), by_band, "recorded", inputs["detections"], inputs["detectionsByBand"])
assert len(rows) == inputs["detections"] and by_band == inputs["detectionsByBand"]

In [ ]:
ref = run["result"]["referenceMjd"]
P = {k: v["value"] for k, v in run["result"]["parameters"].items()}
def bazin(t, log_a, t0, log_tf, log_tr):
    return np.exp(log_a) * np.exp(-(t - t0) / np.exp(log_tf)) / (1 + np.exp(-(t - t0) / np.exp(log_tr)))
data = {}
for (band, _), o in sorted(rows.items(), key=lambda kv: kv[0][1]):
    flux = 10 ** ((23.9 - o["magnitude"]) / 2.5)
    err = flux * math.log(10) / 2.5 * o["magnitude_error"]
    data.setdefault(band, []).append((o["mjd"] - ref, flux, err))
data = {b: np.array(v) for b, v in data.items()}

def residuals(p):
    t0, log_tf, log_tr = p[:3]
    out = []
    for i, band in enumerate(sorted(data)):
        t, f, e = data[band].T
        out.append((bazin(t, p[3 + i], t0, log_tf, log_tr) - f) / e)
    return np.concatenate(out)
# Independent start, not the published solution: peak at the brightest point, 20-day fall,
# 3-day rise, amplitude from each band's brightest flux.
brightest = max((row for band in data for row in data[band]), key=lambda r: r[1])
start = [brightest[0], math.log(20.0), math.log(3.0)] + [math.log(2.5 * data[b][:, 1].max()) for b in sorted(data)]
mine = least_squares(residuals, start, method="trf", x_scale="jac", max_nfev=20000).x
published = np.array([P["t0"], P["log_tau_fall"], P["log_tau_rise"]] + [P["log_amplitude_" + b] for b in sorted(data)])
tol = config["reproduction_tolerance"]
def peak(t0, log_tf, log_tr):
    tf, tr = np.exp(log_tf), np.exp(log_tr)
    return t0 + tr * np.log(tf / tr - 1) if tf > tr else t0
diff = {"peak_days": abs(peak(*mine[:3]) - peak(*published[:3])),
        "tau_fall_fraction": abs(math.exp(mine[1]) / math.exp(published[1]) - 1),
        "tau_rise_fraction": abs(math.exp(mine[2]) / math.exp(published[2]) - 1)}
print({k: float(f"{v:.3g}") for k, v in diff.items()}, "start", [round(x, 3) for x in start[:3]], "tolerances", {k: v for k, v in tol.items() if k != "basis"})
assert diff["peak_days"] <= tol["peak_days"] and diff["tau_fall_fraction"] <= tol["tau_fraction"] and diff["tau_rise_fraction"] <= tol["tau_fraction"]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
grid = np.linspace(min(d[:, 0].min() for d in data.values()) - 10, max(d[:, 0].max() for d in data.values()) + 10, 400)
colors = {"ztfg": "#2a9d8f", "ztfr": "#e76f51", "ztfi": "#6d597a"}
for band in sorted(data):
    t, f, e = data[band].T
    ax.errorbar(t, f, yerr=e, fmt="o", ms=4, color=colors[band], label=f"{band} detections")
    ax.plot(grid, bazin(grid, P["log_amplitude_" + band], P["t0"], P["log_tau_fall"], P["log_tau_rise"]), color=colors[band], lw=1.5,
            label=f"{band} published Bazin model")
ax.set_xlabel(f"days since MJD {ref:.4f}")
ax.set_ylabel("flux density (microjansky, AB zero point 23.9)")
ax.set_title("SN2025bbe: published multi-band Bazin fit (CTAS)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("figure.png", dpi=120)
print("figure.png written")